# Arbitrary bosonic state preparation using smooth pulses : Gradient based dCRAB optimization using GOAT over GRAPE method 


In this example we solve the task in the previous example of preparing arbitrary bosonic state, but using smooth functions as the basis. Here we perform a gradient based dCRAB optimization, by using the *GOAToverGRAPE* (or the GROUP [<span style="color:deepskyblue">Sørensen2018</span>]) method.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from paraqeet.file_logger import FileLogger
from paraqeet.measurement.goat_over_grape import GOATOverGRAPE
from paraqeet.measurement.state_transfer_fidelity import StateTransferFidelityGRAPE
from paraqeet.measurement.weighted_sum_goal import WeightedSumGoal
from paraqeet.model.composite_system import CompositeSystem
from paraqeet.model.coupling import TwoBodyCoupling
from paraqeet.model.qubit import Qubit
from paraqeet.model.resonator import Resonator
from paraqeet.model.rotating_frame import RotatingFrameDrive
from paraqeet.model.schroedinger_equation import SchroedingerEquation
from paraqeet.optimization_map import OptimizationMap
from paraqeet.optimizers.dcrab_optimizer_gradient import DCRABOptimizerGradient
from paraqeet.propagation.scipy_expm_grape import ScipyExpmGRAPE
from paraqeet.quantity import Quantity
from paraqeet.signal.envelopes import DCRABEnvelope
from paraqeet.signal.pwc_generator import PWCGenerator
from paraqeet.signal.waveform import FlatTopGaussianFilter

Following the dCRAB optimization [<span style="color:deepskyblue">Müller2022</span>], we initialize our pulses in the frequency domain by using the `DCRABEnvelope`. Further we use a flat-top Gaussian filter on top to ensure that the pulses start and end at zero. And we use a `PWCGenerator` to generate the piece-wise constant signal required for the `GOAToverGRAPE` method.

In [ ]:
# in seconds (See Eickbusch et al https://arxiv.org/abs/2111.06414 S9 A)
delta_sampling = 33e-9
n_pwc = 40  # number of piecewise constants in the pulse
t_final = n_pwc * delta_sampling  # for now just set up for trying
tlist = np.linspace(0, t_final, n_pwc + 1)
eps_res = 3 * np.pi * 1.0  # initial amplitude of the resonator (in MHz)
eps_max_res = 5 * eps_res  # maximum amplitude of the resonator (in MHz)
eps_qubit = 3 * np.pi * 1.0  # initial amplitude of the qubit (in MHz)
eps_max_qubit = 5 * eps_qubit  # maximum amplitude of the resonator (in MHz)

tone_res = DCRABEnvelope(
    num_components=2,
    max_frequency=2 * np.pi * 2.0,
    amplitude=Quantity(eps_res * 1e6, -eps_max_res * 1e6, eps_max_res * 1e6, name="Amplitude CRAB resonator"),
    t_final=Quantity(t_final, 1 / 2 * t_final, 2 * t_final, name="t_final CRAB resonator"),
    seed=46874,
)

smooth_tone_res = FlatTopGaussianFilter(
    envelopes=tone_res, t_final=Quantity(t_final, 1 / 2 * t_final, 2 * t_final, name="t_final")
)

tone_qubit = DCRABEnvelope(
    num_components=2,
    max_frequency=2 * np.pi * 2.0,
    amplitude=Quantity(eps_qubit * 1e6, -eps_max_qubit * 1e6, eps_max_qubit * 1e6, name="Amplitude CRAB qubit"),
    t_final=Quantity(t_final, 1 / 2 * t_final, 2 * t_final, name="t_final CRAB qubit"),
    seed=189841,
)

smooth_tone_qubit = FlatTopGaussianFilter(
    envelopes=tone_qubit, t_final=Quantity(t_final, 1 / 2 * t_final, 2 * t_final, name="t_final")
)

gen_res = PWCGenerator(envelopes=[smooth_tone_res], tlist=tlist, max_amplitude=2 * eps_max_res * 1e6)
gen_qubit = PWCGenerator(envelopes=[smooth_tone_qubit], tlist=tlist, max_amplitude=2 * eps_max_qubit * 1e6)

Here we plot the resonator tone and compare it to its smooth version. Similarly one can plot the qubit tone.

In [ ]:
from plotting import plot_signal

ts = np.linspace(0, t_final, 1001)
fig, ax = plt.subplots(1, figsize=(5, 3))

plot_signal(smooth_tone_res, ts, ax, linestyle="-", label="Smooth")
plot_signal(gen_res, ts, ax, linestyle="--", label="PWC")

plt.legend()
plt.show()

Like the previous example, we define qubit and resonator parameters, as well as the list of different Fock truncation numbers. We also set the target Fock state.

For now we set the Fock-state truncation at small values (3 and 4) for testing the method. Later we demonstrate the method with (30 and 31) levels in the resonator.

In [ ]:
# Increasing the number of Fock states gives a more realistic scenario, at the price of increasing the simulation time
n_fock_truncation_list = [3, 4]
fock_target = 2
n_times = 1001

omega_range_factor = 1e-3  # just for initialization of the Quantity object
omega_res = 2 * np.pi * 5.26e9
omega_qubit = 2 * np.pi * 6.65e9

omega_drive_res = omega_res
omega_drive_qubit = omega_qubit

detuning_drive_res = omega_res - omega_drive_res
detuning_drive_qubit = omega_qubit - omega_drive_qubit

drive_res = RotatingFrameDrive(gen_res)
drive_qubit = RotatingFrameDrive(gen_qubit)

Similar to the previous example, following [<span style="color:deepskyblue">Heeres2017</span>], we thus consider $N_{\mathrm{T}} \in \{N_{\mathrm{T}}^{(\mathrm{min})}, N_{\mathrm{T}}^{(\mathrm{min})} + 1, \dots, N_{\mathrm{T}}^{(\mathrm{max})} \}$, and introduce a penalty when having different values of fidelities 
for different truncation numbers. Thus, we create different systems, and accordingly fidelity measures, for the different Fock truncation numbers.

In [ ]:
from paraqeet.measurement.utils import overlap_state_vector
from paraqeet.propagation.utils import grape_operator_sandwich_function_closed


def create_experiment(n_fock_truncation_list, fock_target):
    """
    Create the inital and target states, for the given Fock numbers.
    Also create the propagations and measurements for the given truncation numbers and target Fock states.
    """
    resonator_list = []
    coupling_list = []
    model_list = []
    hamiltonian_list = []
    prop_list = []
    initial_state_list = []
    target_state_list = []
    fock_number_op_list = []
    dcrab_fid_list = []

    qubit = Qubit(
        frequency=Quantity(
            value=detuning_drive_qubit,
            min_value=-omega_range_factor * omega_qubit,
            max_value=omega_range_factor * omega_qubit,
        ),
        drives=[drive_qubit],
    )
    ground_state_qubit = np.array([[0.0], [1.0]])

    # We take it to be 10 times the value in Eickbusch et al to speed up the simulation
    chi = 2 * np.pi * 32.81e3 * 10

    for n in n_fock_truncation_list:
        resonator = Resonator(
            frequency=Quantity(
                value=detuning_drive_res,
                min_value=-omega_range_factor * omega_res,
                max_value=omega_range_factor * omega_res,
            ),
            dimension=n,
            drives=[drive_res],
        )
        resonator_list.append(resonator)
        coupling = TwoBodyCoupling(
            resonator,
            qubit,
            is_longitudinal=True,
            coefficient=Quantity(value=chi, min_value=chi / 2, max_value=2 * chi),
        )
        coupling_list.append(coupling)
        ham = CompositeSystem([resonator, qubit], [coupling])
        hamiltonian_list.append(ham)
        model = SchroedingerEquation(
            hamiltonian_func=ham.get_hamiltonian, hamiltonian_and_gradient_func=ham.get_hamiltonian_and_gradient
        )
        model_list.append(model)

        initial_res_state = np.zeros([n, 1], dtype=complex)
        initial_res_state[0, 0] = 1  # vacuum state
        initial_state = np.kron(initial_res_state, ground_state_qubit)
        initial_state_list.append(initial_state)

        target_res_state = np.zeros([n, 1], dtype=complex)
        target_res_state[fock_target, 0] = 1
        target_state = np.kron(target_res_state, ground_state_qubit)
        target_state_list.append(target_state)

        n_op = np.kron(np.diag([x for x in range(n)]), np.identity(2))
        fock_number_op_list.append(n_op)

        # Propagation dt has to be less than `delta_sampling = 33e-9`. Set dt = 30e-9.
        prop = ScipyExpmGRAPE(
            eom_func=model.get_value,
            eom_and_grad_func=model.get_value_and_gradient,
            resolution=1e9,
            initial_state=initial_state,
            target_state=target_state,
            operator_sandwich_function=grape_operator_sandwich_function_closed,
        )
        prop.schirmer_derivative = True

        prop_list.append(prop)

        fid = StateTransferFidelityGRAPE(
            propagation=prop, initial_state=initial_state, target_state=target_state, overlap=overlap_state_vector
        )
        goat_over_grape_fid = GOATOverGRAPE(fid, prop, generators=[gen_res, gen_qubit])
        dcrab_fid_list.append(goat_over_grape_fid)

    return initial_state_list, target_state_list, fock_number_op_list, prop_list, dcrab_fid_list


initial_state_list, target_state_list, fock_number_op_list, prop_list, dcrab_meas_list = create_experiment(
    n_fock_truncation_list, fock_target
)

We consider as cost function of the form

$$
C(\varepsilon(t) ) = w_1 \sum_{N = N_{\mathrm{T}}^{(\mathrm{min})}}^{ N_{\mathrm{T}}^{(\mathrm{max})}} \mathcal{F}_{N} (\varepsilon(t) )   - \frac{w_2}{2} \sum_{N, N' = N_{\mathrm{T}}^{(\mathrm{min})}}^{ N_{\mathrm{T}}^{(\mathrm{max})}} \left[\mathcal{F}_{N}(\varepsilon(t))- \mathcal{F}_{N'} (\varepsilon(t) ) \right]^2,
$$

that we want to maximize. This cost weighted cost function can be constructed using the class WeightedSumGoal. 

*Note - As we consider smooth pulses as our basis function, we do not need to add additional smoothness cost to the goal function.*

In [ ]:
weights = np.array([1.0 for _ in range(len(dcrab_meas_list))])
weight_sum_of_squares = -0.05
total_sum_of_weights = np.sum(weights) + weight_sum_of_squares
weights = weights / total_sum_of_weights
weight_sum_of_squares = weight_sum_of_squares / total_sum_of_weights
dcrab_meas_list_bool = [True for meas in dcrab_meas_list]
sum_of_squares_options = {"weight": weight_sum_of_squares, "meas_bool": dcrab_meas_list_bool}

dcrab_goal = WeightedSumGoal(dcrab_meas_list, weights, sum_of_squares_options=sum_of_squares_options)

We can plot the initial pulses, as well as the corresponding relevant dynamics.

In [ ]:
def plot_states_and_fock_number(item=0):
    """Plot the states."""
    ts = np.linspace(0, t_final, n_times)
    states = prop_list[item].propagate(ts)
    sig_res = gen_res.get_value(ts)
    sig_qubit = gen_qubit.get_value(ts)
    pop_initial_state = (np.abs(initial_state_list[item].conj().T @ states) ** 2).flatten()
    pop_target_state = (np.abs(target_state_list[item].conj().T @ states) ** 2).flatten()

    pop_initial_state = (np.abs(initial_state_list[item].conj().T @ states) ** 2).flatten()

    n_fock_avg = np.zeros(n_times, dtype=float)
    for k in range(n_times):
        n_fock_avg[k] = np.real((states[k].conj().T @ fock_number_op_list[item] @ states[k])[0, 0])

    _, ax = plt.subplots(4, figsize=(4, 10), sharex=True)
    ax[0].plot(ts / 1e-9, np.real(sig_res) / 1e6 / (2 * np.pi), label="I")
    ax[0].plot(ts / 1e-9, np.imag(sig_res) / 1e6 / (2 * np.pi), label="Q")
    ax[0].legend(loc=1)
    ax[0].set_ylabel(r"Field [MHz / $2\pi$]")
    ax[0].set_title("Resonator")
    ax[0].grid(True, linestyle=(1, (1, 5)), linewidth=1)

    ax[1].plot(ts / 1e-9, np.real(sig_qubit) / 1e6 / (2 * np.pi), label="I")
    ax[1].plot(ts / 1e-9, np.imag(sig_qubit) / 1e6 / (2 * np.pi), label="Q")
    ax[1].legend(loc=1)
    ax[1].set_ylabel(r"Field [MHz / $2\pi$]")
    ax[1].set_title("Qubit")
    ax[1].grid(True, linestyle=(1, (1, 5)), linewidth=1)

    ax[2].plot(ts / 1e-9, pop_initial_state, label="$|0,0 \\rangle$")
    ax[2].plot(ts / 1e-9, pop_target_state, label="$|2, 0 \\rangle$")
    ax[2].legend(loc="best")
    ax[2].set_ylabel("Population")
    ax[2].grid(True, linestyle=(1, (1, 5)), linewidth=1)

    ax[3].plot(ts / 1e-9, n_fock_avg)
    ax[3].set_ylabel("$\\langle n \\rangle$")
    if n_fock_truncation_list[item] < 10:
        y_fock_ticks = np.arange(n_fock_truncation_list[item])
    else:
        y_fock_ticks = np.arange(n_fock_truncation_list[item])[::5]
    ax[3].set_yticks(y_fock_ticks)
    ax[3].grid(axis="y", which="major")
    ax[3].minorticks_on()
    ax[3].grid(True, axis="x", linestyle=(1, (1, 5)), linewidth=1)

    ax[-1].set_xlabel("Time [ns]")
    plt.show()


plot_states_and_fock_number()

And we compute the fidelities for the different truncation numbers

In [ ]:
print(f"Fidelity at N_T={n_fock_truncation_list[0]} = {dcrab_meas_list[0].measure(tlist)}")
print(f"Fidelity at N_T={n_fock_truncation_list[1]} = {dcrab_meas_list[1].measure(tlist)}")

which are quite poor! We now proceed with the pulse optimization.

Lets define the parameters we want to optimize. Since, in this case we want to optimize the smooth pulses, the parameters for the optmap would be the parameters of the `DCRABEnvelope` tone.

In [ ]:
params_tone_qubit = tone_qubit.get_parameters()
params_tone_res = tone_res.get_parameters()
params_tone_qubit

Furhter, we can write the optmization progress into a log file, which we can later use to plot the infidelity vs function evaluation.

In [ ]:
import tempfile

temp_dir = tempfile.TemporaryDirectory()
print(f"Logging directory = {temp_dir.name}")

max_iter = 200  # set to 1000 for a good result; set to 10 for a quick example
optmap = OptimizationMap()
optmap.add(tone_res)
optmap.add(tone_qubit)

# Remove the t_final and  DRAG Delta from optimization from drag tone resonator and qubit
optmap.remove(tone_qubit, params_tone_qubit[1])
optmap.remove(tone_res, params_tone_res[1])
optmap.register_params_with_optimizables()

file_logger = FileLogger(temp_dir.name)

# Define the optimizer
opt = DCRABOptimizerGradient(
    dcrab_goal,
    optimization_map=optmap,
    super_iteration_every=100,
    max_super_iteration_num=2,
    print_every_iteration_num=50,
    super_iteration_tol=1e-6,
    seed=5648,
)
opt.set_options({"maxfun": max_iter, "workers": 32, "ftol": 1e-4})
opt.logger = file_logger

In [ ]:
optmap

In [ ]:
opt.optimize(gen_res.tlist)

Lets set the optimization to the best parameters obtained during the run

In [ ]:
opt.set_parameters(opt.best_params)

We can now plot the new pulses and the corresponding dynamics. If the number of iterations (max_iter) is chosen to be large enough, we should see an improvement.

In [ ]:
plot_states_and_fock_number()

We can also compare the dynamics with the higher truncation number. Ideally, if the pulse amplitude is not high enough to produce artifacts due to the truncation of Hilbert space, the dyanmics in the two cases should match each other. Else one needs to either reduce the pulse amplitude or increase the truncation cutoff.

In [ ]:
plot_states_and_fock_number(1)

The new fidelities are

In [ ]:
print(f"Fidelity at N_T={n_fock_truncation_list[0]} = {dcrab_meas_list[0].measure(tlist)}")
print(f"Fidelity at N_T={n_fock_truncation_list[1]} = {dcrab_meas_list[1].measure(tlist)}")

For this small truncation number the dynamics and fidelities do not match very well. This indicates the need for higher truncation numbers, which we demonstrate in the next section.

Finally, lets plot the variation of infidelity with evaluation number.

In [ ]:
from plotting import plot_infidelity_vs_evaluation_from_logs

plot_infidelity_vs_evaluation_from_logs(log_path=temp_dir.name + "/opt.log", label="dCRAB optimization");

## Setting trunction to higher value

Here we set the truncation values to 30 and 31 levels for a more realistic simualtion and rerun the entire simulation.

*Note - The following takes about 60 mins to run on an AMD-EPYC Milan processor with 128 cores and 256GB RAM (might take more depending on the number of CPU cores available).*

We first reset the generator parameters (by redefining them), and redefine the resonator with higher truncation numbers.

In [ ]:
# in seconds (See Eickbusch et al https://arxiv.org/abs/2111.06414 S9 A)
delta_sampling = 33e-9
n_pwc = 40  # number of piecewise constants in the pulse
t_final = n_pwc * delta_sampling  # for now just set up for trying
tlist = np.linspace(0, t_final, n_pwc + 1)
eps_res = 3 * np.pi * 1.0  # initial amplitude of the resonator (in MHz)
eps_max_res = 5 * eps_res  # maximum amplitude of the resonator (in MHz)
eps_qubit = 3 * np.pi * 1.0  # initial amplitude of the qubit (in MHz)
eps_max_qubit = 5 * eps_qubit  # maximum amplitude of the resonator (in MHz)

tone_res = DCRABEnvelope(
    num_components=3,
    max_frequency=2 * np.pi * 5.0,
    amplitude=Quantity(eps_res * 1e6, -eps_max_res * 1e6, eps_max_res * 1e6, name="Amplitude CRAB resonator"),
    t_final=Quantity(t_final, 1 / 2 * t_final, 2 * t_final, name="t_final CRAB resonator"),
    seed=17619276394480986,
)

smooth_tone_res = FlatTopGaussianFilter(
    envelopes=tone_res, t_final=Quantity(t_final, 1 / 2 * t_final, 2 * t_final, name="t_final")
)

tone_qubit = DCRABEnvelope(
    num_components=3,
    max_frequency=2 * np.pi * 5.0,
    amplitude=Quantity(eps_qubit * 1e6, -eps_max_qubit * 1e6, eps_max_qubit * 1e6, name="Amplitude CRAB qubit"),
    t_final=Quantity(t_final, 1 / 2 * t_final, 2 * t_final, name="t_final CRAB qubit"),
    seed=17619276394849436,
)

smooth_tone_qubit = FlatTopGaussianFilter(
    envelopes=tone_qubit, t_final=Quantity(t_final, 1 / 2 * t_final, 2 * t_final, name="t_final")
)

gen_res = PWCGenerator(envelopes=[smooth_tone_res], tlist=tlist, max_amplitude=2 * eps_max_res * 1e6)
gen_qubit = PWCGenerator(envelopes=[smooth_tone_qubit], tlist=tlist, max_amplitude=2 * eps_max_qubit * 71e6)

In [ ]:
# Increasing the number of Fock states gives a more realistic scenario, at the price of increasing the simulation time
n_fock_truncation_list = [30, 31]
fock_target = 2
n_times = 1001

omega_range_factor = 1e-3  # just for initialization of the Quantity object
omega_res = 2 * np.pi * 5.26e9
omega_qubit = 2 * np.pi * 6.65e9

omega_drive_res = omega_res
omega_drive_qubit = omega_qubit

detuning_drive_res = omega_res - omega_drive_res
detuning_drive_qubit = omega_qubit - omega_drive_qubit

drive_res = RotatingFrameDrive(gen_res)
drive_qubit = RotatingFrameDrive(gen_qubit)


# Create a new experiment and redefine the measurement
initial_state_list, target_state_list, fock_number_op_list, prop_list, dcrab_meas_list = create_experiment(
    n_fock_truncation_list, fock_target
)

weights = np.array([1.0 for _ in range(len(dcrab_meas_list))])
weight_sum_of_squares = -0.05
total_sum_of_weights = np.sum(weights) + weight_sum_of_squares
weights = weights / total_sum_of_weights
weight_sum_of_squares = weight_sum_of_squares / total_sum_of_weights
dcrab_meas_list_bool = [True for meas in dcrab_meas_list]
sum_of_squares_options = {"weight": weight_sum_of_squares, "meas_bool": dcrab_meas_list_bool}

dcrab_goal = WeightedSumGoal(dcrab_meas_list, weights, sum_of_squares_options=sum_of_squares_options)

In [ ]:
plot_states_and_fock_number()

In [ ]:
plot_states_and_fock_number(1)

Initial fidelity before optimization

In [ ]:
print(f"Fidelity at N_T={n_fock_truncation_list[0]} = {dcrab_meas_list[0].measure(tlist)}")
print(f"Fidelity at N_T={n_fock_truncation_list[1]} = {dcrab_meas_list[1].measure(tlist)}")

In [ ]:
tone_res.get_parameters()

In [ ]:
tone_qubit.get_parameters()

Redefine the optmap and the optmizer and rerun the optimization

In [ ]:
import tempfile

temp_dir = tempfile.TemporaryDirectory()
print(f"Logging directory = {temp_dir.name}")

max_iter = 1000  # set to 1000 for a good result; set to 10 for a quick example
optmap = OptimizationMap()
optmap.add(tone_res)
optmap.add(tone_qubit)

# Remove the t_final and  DRAG Delta from optimization from drag tone resonator and qubit
optmap.remove(tone_qubit, params_tone_qubit[1])
optmap.remove(tone_res, params_tone_res[1])
optmap.register_params_with_optimizables()

file_logger = FileLogger(temp_dir.name)

# Define the optimizer
opt = DCRABOptimizerGradient(
    dcrab_goal,
    optimization_map=optmap,
    super_iteration_every=500,
    max_super_iteration_num=3,
    print_every_iteration_num=100,
    super_iteration_tol=1e-6,
    seed=8647,
)
opt.set_options({"maxfun": max_iter, "workers": 32})
opt.logger = file_logger

In [ ]:
opt.optimize(gen_res.tlist)
opt.set_parameters(opt.best_params)

And the new fidelities are

In [ ]:
print(f"Fidelity at N_T={n_fock_truncation_list[0]} = {dcrab_meas_list[0].measure(tlist)}")
print(f"Fidelity at N_T={n_fock_truncation_list[1]} = {dcrab_meas_list[1].measure(tlist)}")

Here the dynamics and the fidelities are very similar to one another, indicating no truncation artifacts.

Lets plot the optimized pulses and dyanmics

In [ ]:
plot_states_and_fock_number()

And verify the dynamics with the higher truncation looks the same

In [ ]:
plot_states_and_fock_number(1)

Finally, lets plot the variation of infidelity with evaluation number.

In [ ]:
from plotting import plot_infidelity_vs_evaluation_from_logs

plot_infidelity_vs_evaluation_from_logs(log_path=temp_dir.name + "/opt.log", label="dCRAB optimization");

The sharp rise infidelity represents the beginning of a super-iteration

## <em>References</em>

[<span style="color:deepskyblue">Sørensen2018</span>]Sørensen, Jens Jakob WH, et al. "Quantum optimal control in a chopped basis: Applications in control of Bose-Einstein condensates." Physical Review A 98.2 (2018): 022119. \
[<span style="color:deepskyblue">Müller2022</span>] Matthias M Müller et al "One decade of quantum optimal control in the chopped random basis", Rep. Prog. Phys. 85 076001 (2022) \
[<span style="color:deepskyblue">Heeres2017</span>] R. Heeres et al., "<em>Implementing a universal gate set on a logical qubit encoded in an oscillator</em>", Nature Communications 8, 94 (2017)